# 08 — Text-to-Image Pipelines

Stable Diffusion decomposes into components that can each be analysed and modified independently.

## Stable Diffusion Architecture Walkthrough

Stable Diffusion (v1.5, v2.1, SDXL) combines:

1. **CLIP text encoder**: transforms the text prompt into a sequence of token embeddings. The cross-attention layers in the UNet use these as keys and values, allowing the model to attend to relevant text tokens at each denoising step.

2. **VAE encoder/decoder**: compresses 512×512 images to 64×64×4 latents (8x spatial compression). The decoder adds back high-frequency detail. KL-regularised to prevent posterior collapse.

3. **UNet with cross-attention**: the core denoiser. Alternates between:
   - Residual conv blocks (spatial processing)
   - Transformer blocks with self-attention (spatial context) and cross-attention (text conditioning)
   - Time-step conditioning via AdaGN (adaptive group normalisation with time embedding)

4. **DDIM/PLMS/DPM++ scheduler**: the inference sampler. DDIM is simple; DPM++ is faster for the same quality.

**Prompt mechanics**:
- Long prompts are truncated at 77 tokens (CLIP context limit)
- Negative prompts are handled via CFG: unconditioned = negative prompt
- Token weights (e.g., `cat:1.5`) modify attention scaling
- `<|endoftext|>` tokens fill unused positions

In [ ]:
# SD pipeline decomposition and component analysis
import numpy as np
import torch
import torch.nn as nn

# Component 1: Simple text encoder (CLIP-like)
class SimpleCLIPTextEncoder(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=64, seq_len=16, n_layers=2):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=128, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.seq_len = seq_len

    def forward(self, token_ids):
        pos = torch.arange(token_ids.shape[1]).unsqueeze(0)
        h = self.token_embed(token_ids) + self.pos_embed(pos)
        return self.transformer(h)  # (batch, seq, embed_dim)

text_encoder = SimpleCLIPTextEncoder()
prompt_tokens = torch.randint(0, 1000, (1, 16))
text_emb = text_encoder(prompt_tokens)
print(f'Text embedding shape: {text_emb.shape}')  # (1, 16, 64)

# Component 2: Cross-attention UNet block
class CrossAttentionBlock(nn.Module):
    def __init__(self, spatial_dim=32, text_dim=64, n_heads=4):
        super().__init__()
        self.norm = nn.LayerNorm(spatial_dim)
        self.q = nn.Linear(spatial_dim, spatial_dim)
        self.k = nn.Linear(text_dim, spatial_dim)
        self.v = nn.Linear(text_dim, spatial_dim)
        self.out = nn.Linear(spatial_dim, spatial_dim)
        self.n_heads = n_heads

    def forward(self, spatial_feat, text_feat):
        # spatial_feat: (B, H*W, C); text_feat: (B, T, D)
        B, N, C = spatial_feat.shape
        q = self.q(self.norm(spatial_feat))
        k = self.k(text_feat)
        v = self.v(text_feat)
        # Scaled dot-product attention
        attn = torch.bmm(q, k.transpose(1, 2)) / C**0.5
        attn = attn.softmax(dim=-1)
        out = torch.bmm(attn, v)
        return spatial_feat + self.out(out)

cross_attn = CrossAttentionBlock()
spatial = torch.randn(1, 16, 32)  # 4x4 feature map flattened
out = cross_attn(spatial, text_emb)
print(f'Cross-attention output: {out.shape}')

# Inspect attention patterns
with torch.no_grad():
    q = cross_attn.q(cross_attn.norm(spatial))
    k = cross_attn.k(text_emb)
    attn_weights = (q @ k.transpose(1, 2) / 32**0.5).softmax(dim=-1)
print(f'Attention map shape (spatial → text): {attn_weights.shape}')
print(f'Each spatial position attends to {attn_weights.shape[2]} text tokens')

In [ ]:
# Pipeline decomposition analysis
import matplotlib.pyplot as plt

# Simulate the SD inference pipeline stages
components = [
    ('CLIP Text Encoder', 123, 77, 'Text tokens → 768-dim embeddings'),
    ('DDIM Scheduler', 2, 50, '50 reverse steps, DDIM with eta=0'),
    ('VAE Encoder', 83, None, 'Image → 64x64x4 latent (training only)'),
    ('UNet Denoiser', 860, None, '2D UNet with cross-attention; 1.0B params in SD 1.5'),
    ('VAE Decoder', 83, None, '64x64x4 latent → 512x512 image'),
]

print('Stable Diffusion Pipeline Decomposition')
print('='*70)
print(f'{"Component":<22} {"Params (M)":>12} {"Notes"}')
print('-'*70)
for name, params, steps, notes in components:
    print(f'{name:<22} {params:>12} {notes}')

print('\nInference compute breakdown:')
print('  CLIP: 1x forward pass')
print('  UNet: 50 forward passes (DDIM 50 steps, or 2x per CFG step = 100 total)')
print('  VAE decode: 1x forward pass')
print('  Total: ~100 UNet passes dominate compute (~98% of inference time)')

# CFG doubles UNet calls
print('\nWith CFG (guidance_scale=7.5):')
print('  Each step: 2 UNet calls (conditioned + unconditioned)')
print('  50 DDIM steps × 2 = 100 UNet forward passes')